# LABORATORIO 2 – ANALIZADOR SINTÁCTICO PARA MINIJAVA

**Asignatura:** Compiladores / Procesadores de Lenguaje  
**Estudiante:** Maria Alejandra Arias Medina  
**NRC:** 2439  
**Tecnología:** Python y Pyparsing

**Objetivo:** implementar un analizador sintáctico para MiniJava que valide la estructura de un programa fuente, construya un árbol sintáctico mediante listas anidadas y reporte errores sintácticos.

> Antes de entregar, confirma que el nombre, NRC y datos de identificación coincidan con los solicitados por el profesor y con la conformación real del grupo.

# Gramática libre de contexto de MiniJava

## 1. Convenciones

La gramática se expresa en notación BNF/EBNF. Los símbolos entre `< >` son no terminales; los terminales se escriben entre comillas o mediante el nombre del token léxico. `ε` representa la cadena vacía y `|` representa alternativas.

**Símbolo inicial:** `<programa>`.

## 2. Conjuntos

**No terminales:**

\[
N=\{<programa>,<sentencias>,<sentencia>,<bloque>,<declaracion>,<asignacion>,<incremento>,<impresion>,<condicional>,<while>,<dowhile>,<for>,<switch>,<caso>,<default>,<expresion>,<expresion\_unaria>,<primario>,<literal>,<tipo>\}
\]

**Terminales (tokens del analizador léxico MiniJava):**

\[
T=\{INT\_TIPO,FLOAT\_TIPO,BOOLEAN\_TIPO,BOOL\_TIPO,CHAR\_TIPO,STRING\_TIPO,ID,INT\_LITERAL,FLOAT\_LITERAL,STRING\_LITERAL,CHAR\_LITERAL,TRUE,FALSE,IF,ELSE,WHILE,DO,FOR,SWITCH,CASE,DEFAULT,BREAK,CONTINUE,SYSTEM,OUT,PRINT,PRINTLN,ASIGN,MAS\_ASIGN,MENOS\_ASIGN,MULT\_ASIGN,DIV\_ASIGN,INCREMENT,DECREMENT,MAS,MENOS,MULT,DIV,MOD,EQ,NE,LT,GT,LE,GE,AND,OR,NOT,PARENI,PAREND,LLAVEI,LLAVED,PUNTOYCOMA,COMA,PUNTO,DOSPUNTOS\}
\]

## 3. Producciones

\[
\begin{aligned}
<programa> &\rightarrow <sentencias> \\
<sentencias> &\rightarrow <sentencia><sentencias>\mid\varepsilon \\
<sentencia> &\rightarrow <declaracion>\mid<asignacion>\mid<incremento>\mid<impresion>\\
&\mid<condicional>\mid<while>\mid<dowhile>\mid<for>\mid<switch>\\
&\mid BREAK\ PUNTOYCOMA\mid CONTINUE\ PUNTOYCOMA\mid<bloque>\\
<bloque> &\rightarrow LLAVEI\ <sentencias>\ LLAVED \\
<tipo> &\rightarrow INT\_TIPO\mid FLOAT\_TIPO\mid BOOLEAN\_TIPO\mid CHAR\_TIPO\mid STRING\_TIPO\\
<declaracion> &\rightarrow <tipo>\ ID\ [ASIGN\ <expresion>]\ PUNTOYCOMA\\
<asignacion> &\rightarrow ID\ (ASIGN\mid MAS\_ASIGN\mid MENOS\_ASIGN\mid MULT\_ASIGN\mid DIV\_ASIGN)\ <expresion>\ PUNTOYCOMA\\
<incremento> &\rightarrow ID\ (INCREMENT\mid DECREMENT)\ PUNTOYCOMA\\
<impresion> &\rightarrow SYSTEM\ PUNTO\ OUT\ PUNTO\ (PRINT\mid PRINTLN)\ PARENI\ <expresion>\ PAREND\ PUNTOYCOMA\\
<condicional> &\rightarrow IF\ PARENI\ <expresion>\ PAREND\ <bloque>\ [ELSE\ (<bloque>\mid<condicional>)]\\
<while> &\rightarrow WHILE\ PARENI\ <expresion>\ PAREND\ <bloque>\\
<dowhile> &\rightarrow DO\ <bloque>\ WHILE\ PARENI\ <expresion>\ PAREND\ PUNTOYCOMA\\
<for> &\rightarrow FOR\ PARENI\ <inicio>\ PUNTOYCOMA\ <expresion>\ PUNTOYCOMA\ [<paso>]\ PAREND\ <bloque>\\
<inicio> &\rightarrow <tipo>\ ID\ ASIGN\ <expresion>\mid ID\ ASIGN\ <expresion>\\
<paso> &\rightarrow ID\ (INCREMENT\mid DECREMENT)\mid ID\ (ASIGN\mid MAS\_ASIGN\mid MENOS\_ASIGN\mid MULT\_ASIGN\mid DIV\_ASIGN)\ <expresion>\\
<switch> &\rightarrow SWITCH\ PARENI\ <expresion>\ PAREND\ LLAVEI\ <casos>\ [<default>]\ LLAVED\\
<casos> &\rightarrow <caso><casos>\mid\varepsilon\\
<caso> &\rightarrow CASE\ <literal>\ DOSPUNTOS\ <sentencias>\\
<default> &\rightarrow DEFAULT\ DOSPUNTOS\ <sentencias>
\end{aligned}
\]

## 4. Expresiones y precedencia

\[
\begin{aligned}
<expresion> &\rightarrow <expresion\_logica\_or>\\
<expresion\_logica\_or> &\rightarrow <expresion\_logica\_and>\ (OR\ <expresion\_logica\_and>)*\\
<expresion\_logica\_and> &\rightarrow <expresion\_relacional>\ (AND\ <expresion\_relacional>)*\\
<expresion\_relacional> &\rightarrow <expresion\_aditiva>\ ((LT\mid LE\mid GT\mid GE\mid EQ\mid NE)\ <expresion\_aditiva>)*\\
<expresion\_aditiva> &\rightarrow <expresion\_multiplicativa>\ ((MAS\mid MENOS)\ <expresion\_multiplicativa>)*\\
<expresion\_multiplicativa> &\rightarrow <expresion\_unaria>\ ((MULT\mid DIV\mid MOD)\ <expresion\_unaria>)*\\
<expresion\_unaria> &\rightarrow (NOT\mid MENOS)\ <expresion\_unaria>\mid<primario>\\
<primario> &\rightarrow ID\mid INT\_LITERAL\mid FLOAT\_LITERAL\mid STRING\_LITERAL\\
&\mid CHAR\_LITERAL\mid TRUE\mid FALSE\mid PARENI\ <expresion>\ PAREND
\end{aligned}
\]

| Nivel | Operadores | Asociatividad |
|---|---|---|
| 1 (mayor) | `!`, `-` unarios | Derecha |
| 2 | `*`, `/`, `%` | Izquierda |
| 3 | `+`, `-` | Izquierda |
| 4 | `<`, `<=`, `>`, `>=`, `==`, `!=` | Izquierda |
| 5 | `&&` | Izquierda |
| 6 (menor) | `||` | Izquierda |

## 5. Resolución del *dangling else*

El `else` se asocia al `if` no emparejado más cercano. En la implementación recursiva, la alternativa de `else` puede contener otro `<condicional>`; por ello, un `else if` se interpreta como un `if` anidado y no como una alternativa independiente. Los bloques delimitados por llaves eliminan cualquier ambigüedad adicional.

## 6. Compatibilidad con los tokens del Laboratorio 1

El analizador reconoce los lexemas reservados y operadores del lenguaje definido en el enunciado. También admite los nombres de tipos y la forma `print(expresion);` que aparecen en el analizador léxico entregado en el Laboratorio 1 (`bool`, `string` y `print`), además de las formas MiniJava solicitadas aquí (`boolean`, `String`, `System.out.print` y `System.out.println`). Esto permite probar los ejemplos del enunciado y los lexemas de la práctica léxica.

## 7. Restricciones del lenguaje

- `for` admite una sola variable de inicio; no admite declaración múltiple separada por comas.
- El paso de `for` puede omitirse. En ese caso, la implementación representa el paso como `1` para comparaciones crecientes y `-1` para comparaciones decrecientes.
- `switch` admite `case` con literales y un `default` opcional al final.
- No se incluyen clases, métodos, arreglos ni funciones de usuario.
- Los comentarios `//` y `/* ... */` y los espacios en blanco se ignoran.


# Manual de uso

## 1. Requisitos

- Python 3.
- Librería `pyparsing`.
- Google Colab o un entorno local con Python.
- Archivo fuente del programa con extensión `.txt`.

## 2. Instalación

En Google Colab, si Pyparsing no está disponible, ejecutar en una celda antes del código:

```python
!pip install pyparsing
```

## 3. Ejecución en Google Colab

1. Ejecutar la celda 4 completa.
2. Cuando aparezca el selector de archivos, cargar el programa MiniJava guardado como `.txt`.
3. El programa leerá el archivo en UTF-8 y analizará su contenido.
4. Revisar la salida: `Codigo Valido` o `Errores de sintaxis`, el listado de errores y el árbol de análisis sintáctico.

## 4. Ejecución local

1. Instalar Pyparsing desde la terminal: `python -m pip install pyparsing`.
2. Guardar el código de la celda 4 en un archivo `.py`.
3. Ejecutarlo con `python nombre_del_archivo.py`.
4. Escribir la ruta del archivo `.txt` cuando el programa la solicite.

## 5. Elementos reconocidos

Declaraciones con tipos `int`, `float`, `boolean`/`bool`, `char` y `String`/`string`; asignaciones simples y compuestas; incremento/decremento; expresiones aritméticas, relacionales y lógicas; bloques; `if-else`; `while`; `do-while`; `for`; `switch-case-default`; `System.out.print`; `System.out.println`; `print`; `break` y `continue`.

## 6. Formato de salida

El árbol es una lista de listas. Los bloques contienen directamente la lista de sentencias, sin un nivel artificial adicional. Los operadores de una misma precedencia se conservan en notación infija. Los literales de texto mantienen sus comillas.

**Importante:** el analizador valida sintaxis; no ejecuta el programa ni comprueba tipos, variables declaradas o semántica. Cuando encuentra errores, registra su línea y contexto, inserta nodos `error` en el árbol cuando corresponde y continúa con las sentencias posteriores. La rúbrica exige que los archivos de prueba se carguen de forma interactiva y que el análisis no dependa del nombre del archivo.

In [ ]:
import pyparsing as pp
import re
pp.ParserElement.enable_packrat()
pp.ParserElement.set_default_whitespace_chars(" \t\r\n")

class Nodo:
    def __init__(self, valor): self.valor = valor
    def __repr__(self): return repr(self.valor)

def valor(x):
    if isinstance(x, Nodo): return valor(x.valor)
    if isinstance(x, pp.ParseResults): return [valor(v) for v in x]
    if isinstance(x, list): return [valor(v) for v in x]
    return x

def lista(x):
    return valor(x) if isinstance(x, (Nodo, pp.ParseResults)) else [valor(v) for v in x]

TIPOS = ("int", "float", "boolean", "bool", "char", "String", "string")
kw = lambda s: pp.Keyword(s)

# Comentarios se eliminan antes del analisis sintactico.
COMENTARIOS = pp.Regex(r"//[^\n]*|/\*[\s\S]*?\*/")

ident = pp.Regex(r"[A-Za-z][A-Za-z0-9]*")
numero = pp.Regex(r"(?:\d+\.\d+(?:[eE][+-]?\d+)?|\d+[eE][+-]?\d+|\d+)")
string_lit = pp.Regex(r'"(?:[^"\\\n]|\\.)*"')
char_lit = pp.Regex(r"'(?:[^'\\\n]|\\.)'")
boolean_lit = pp.MatchFirst([kw("true"), kw("false")])
atom = pp.MatchFirst([numero, string_lit, char_lit, boolean_lit, ident])

# Expresiones: infix_notation aplica la precedencia indicada en el enunciado.
expr = pp.Forward()
parens = pp.Suppress("(") + expr + pp.Suppress(")")
base = pp.MatchFirst([parens, atom])
def expr_unaria(t): return Nodo([valor(t[0][0]), valor(t[0][1])])
def expr_binaria(t): return Nodo(lista(t[0]))
expr <<= pp.infix_notation(base, [
    (pp.MatchFirst([pp.Literal("!"), pp.Literal("-")]), 1, pp.OpAssoc.RIGHT, expr_unaria),
    (pp.MatchFirst([pp.Literal("*"), pp.Literal("/"), pp.Literal("%")]), 2, pp.OpAssoc.LEFT, expr_binaria),
    (pp.MatchFirst([pp.Literal("+"), pp.Literal("-")]), 2, pp.OpAssoc.LEFT, expr_binaria),
    (pp.MatchFirst([pp.Literal("<="), pp.Literal(">="), pp.Literal("<"), pp.Literal(">"), pp.Literal("=="), pp.Literal("!=")]), 2, pp.OpAssoc.LEFT, expr_binaria),
    (pp.Literal("&&"), 2, pp.OpAssoc.LEFT, expr_binaria),
    (pp.Literal("||"), 2, pp.OpAssoc.LEFT, expr_binaria),
])
expr.set_parse_action(lambda t: Nodo(valor(t[0])))

LP, RP = pp.Suppress("("), pp.Suppress(")")
LB, RB = pp.Suppress("{"), pp.Suppress("}")
SEMI, COLON = pp.Suppress(";"), pp.Suppress(":")
tipo = pp.MatchFirst([kw(t) for t in TIPOS])

sentencia = pp.Forward()
bloque = pp.Forward()

# Declaracion y asignacion.
declaracion = (tipo + ident + pp.Optional(pp.Suppress("=") + expr) + SEMI)
def pa_decl(t):
    r = [t[0], t[1]]
    if len(t) == 3:
        e = valor(t[2]); r += e if isinstance(e, list) else [e]
    return Nodo(r)
declaracion.set_parse_action(pa_decl)

op_asig = pp.MatchFirst([pp.Literal(x) for x in ("+=", "-=", "*=", "/=", "=")])
asignacion = (ident + op_asig + expr + SEMI)
def pa_asig(t):
    e = valor(t[2]); return Nodo([t[0], t[1]] + (e if isinstance(e, list) else [e]))
asignacion.set_parse_action(pa_asig)
incremento = (ident + pp.MatchFirst([pp.Literal("++"), pp.Literal("--")]) + SEMI).set_parse_action(lambda t: Nodo([t[0], t[1]]))

# print / println.
impresion_java = (pp.Suppress(kw("System")) + pp.Suppress(".") + pp.Suppress(kw("out")) + pp.Suppress(".") + pp.MatchFirst([kw("print"), kw("println")]) + LP + expr + RP + SEMI)
impresion_simple = (pp.Suppress(kw("print")) + LP + expr + RP + SEMI).set_parse_action(lambda t: ["print"] + list(t))
impresion = pp.MatchFirst([impresion_java, impresion_simple])
def pa_print(t):
    e = valor(t[1]); return Nodo([t[0]] + (e if isinstance(e, list) else [e]))
impresion.set_parse_action(pa_print)
break_stmt = (pp.Suppress(kw("break")) + SEMI).set_parse_action(lambda: Nodo(["break"]))
continue_stmt = (pp.Suppress(kw("continue")) + SEMI).set_parse_action(lambda: Nodo(["continue"]))

# Bloques: no agregan un nivel artificial al arbol.
bloque <<= (LB + pp.Group(pp.ZeroOrMore(sentencia)) + RB).set_parse_action(lambda t: Nodo(lista(t[0])))

# if/else: la recursion consume primero el else mas cercano (dangling else).
if_stmt = pp.Forward()
else_stmt = pp.Forward()
else_stmt <<= pp.MatchFirst([if_stmt, bloque])
def pa_if(t):
    r = ["if", valor(t[0]), valor(t[1])]
    if len(t) > 2: r.append(valor(t[2]))
    return Nodo(r)
if_stmt <<= (pp.Suppress(kw("if")) + LP + expr + RP + bloque + pp.Optional(pp.Suppress(kw("else")) + else_stmt)).set_parse_action(pa_if)
while_stmt = (pp.Suppress(kw("while")) + LP + expr + RP + bloque).set_parse_action(lambda t: Nodo(["while", valor(t[0]), valor(t[1])]))
do_stmt = (pp.Suppress(kw("do")) + bloque + pp.Suppress(kw("while")) + LP + expr + RP + SEMI).set_parse_action(lambda t: Nodo(["dowhile", valor(t[0]), valor(t[1])]))

# for: inicio puede declarar o reutilizar variable; paso opcional con incremento inferido.
inicio_decl = tipo + ident + pp.Suppress("=") + expr
inicio_asig = ident + pp.Suppress("=") + expr
inicio = pp.MatchFirst([inicio_decl, inicio_asig])
paso = pp.MatchFirst([
    ident + pp.MatchFirst([pp.Literal("++"), pp.Literal("--")]),
    ident + op_asig + expr,
    ident + pp.Suppress("=") + expr,
])
for_stmt = (pp.Suppress(kw("for")) + LP + pp.Group(inicio) + SEMI + expr + SEMI + pp.Optional(pp.Group(paso)) + RP + bloque)
def pa_for(t):
    ini = lista(t[0])
    if len(ini) == 3: tipo_v, var, inicial = ini
    else: tipo_v, var, inicial = None, ini[0], ini[1]
    cond = valor(t[1])
    if len(t) == 4:
        p = valor(t[2])
        if len(p) == 2: paso_v = p
        else:
            e = p[2]; paso_v = [p[0], p[1]] + (e if isinstance(e, list) else [e])
    else:
        op = next((x for x in (cond if isinstance(cond, list) else []) if x in ("<", "<=", ">", ">=")), "<")
        paso_v = [var, "-1" if op in (">", ">=") else "1"]
    cab = ([tipo_v] if tipo_v else []) + [var, valor(inicial), cond, paso_v]
    return Nodo(["for", cab, valor(t[-1])])
for_stmt.set_parse_action(pa_for)

# switch-case: case acepta solo literales y default es opcional al final.
literal_case = pp.MatchFirst([numero, string_lit, char_lit, boolean_lit])
case_stmt = (pp.Suppress(kw("case")) + literal_case + pp.Optional(pp.Literal(":")) + pp.Group(pp.ZeroOrMore(~(kw("case") | kw("default") | RB) + sentencia)))
def pa_case(s, loc, t):
    linea = pp.lineno(loc, s)
    if str(t[0]) == '"__CASE_COLON_MISSING__"':
        ERRORES.append((linea, "case", "faltan dos puntos"))
        return Nodo(["case", ["error", "case", "faltan dos puntos"], lista(t[-1])])
    if len(t) >= 3 and t[1] == ":":
        return Nodo(["case", t[0], lista(t[2])])
    ERRORES.append((linea, "case", "faltan dos puntos"))
    return Nodo(["case", ["error", "case", "faltan dos puntos"], lista(t[-1])])
case_stmt.set_parse_action(pa_case)
default_stmt = (pp.Suppress(kw("default")) + COLON + pp.Group(pp.ZeroOrMore(~RB + sentencia))).set_parse_action(lambda t: Nodo(["default", lista(t[0])]))
switch_stmt = (pp.Suppress(kw("switch")) + LP + expr + RP + LB + pp.Group(pp.ZeroOrMore(case_stmt)) + pp.Optional(default_stmt) + RB).set_parse_action(lambda t: Nodo(["switch", valor(t[0]), valor(t[1]) + ([valor(t[2])] if len(t) > 2 else [])]))

# Recuperacion especifica para encabezados de control mal formados.
ERRORES = []

def analizar_cabecera_control(s, loc, t, nombre):
    encabezado = t[0]
    cuerpo = valor(t[1])
    linea = pp.lineno(loc, s)
    contenido = encabezado[encabezado.find("(") + 1:]
    tiene_cierre = ")" in contenido
    if tiene_cierre:
        contenido = contenido[:contenido.rfind(")")].strip()
    else:
        contenido = contenido.strip()
    if nombre == "for":
        if not tiene_cierre:
            contexto, desc = "for", "falta parentesis de cierre"
        elif contenido.count(";") != 2:
            contexto, desc = "for", "faltan separadores ';' en el encabezado"
        else:
            contexto, desc = "for", "encabezado mal formado"
        nodo = ["error", contexto, desc]
        ERRORES.append((linea, contexto, desc))
        return Nodo(["for", nodo, cuerpo])
    if not tiene_cierre:
        contexto, desc = nombre, "falta parentesis de cierre"
        nodo = ["error", contexto, desc]
    elif not contenido:
        contexto, desc = "condicion", "condicion vacia"
        nodo = ["error", contexto, desc]
    else:
        try:
            expr.parse_string(contenido, parse_all=True)
            contexto, desc = nombre, "encabezado mal formado"
        except pp.ParseBaseException:
            contexto, desc = "condicion", "expresion incompleta"
        nodo = ["error", contexto, desc]
    ERRORES.append((linea, contexto, desc))
    salida = [nombre, nodo, cuerpo]
    if nombre == "if" and len(t) > 2:
        salida.append(valor(t[2]))
    return Nodo(salida)

cabecera_if_mala = pp.Regex(r"if\s*\([^{}]*?(?:\)|(?=\{))\s*(?=\{)") + bloque + pp.Optional(pp.Suppress(kw("else")) + else_stmt)
cabecera_if_mala.set_parse_action(lambda s, l, t: analizar_cabecera_control(s, l, t, "if"))
cabecera_while_mala = pp.Regex(r"while\s*\([^{}]*?(?:\)|(?=\{))\s*(?=\{)") + bloque
cabecera_while_mala.set_parse_action(lambda s, l, t: analizar_cabecera_control(s, l, t, "while"))
cabecera_for_mala = pp.Regex(r"for\s*\([^{}]*?(?:\)|(?=\{))\s*(?=\{)") + bloque
cabecera_for_mala.set_parse_action(lambda s, l, t: analizar_cabecera_control(s, l, t, "for"))

# Recupera do-while cuando falta el punto y coma obligatorio final.
do_malo = pp.Regex(r"do\s*\{[\s\S]*?\}\s*while\s*\([^{}]*\)")
def pa_do_malo(s, loc, t):
    bruto = t[0]
    linea = pp.lineno(loc, s)
    cuerpo_txt = re.search(r"do\s*\{([\s\S]*?)\}\s*while", bruto).group(1)
    cuerpo_normalizado = re.sub(r"(?m)(\b[A-Za-z][A-Za-z0-9]*\s*(?:\+\+|--))\s*$", r"\1;", cuerpo_txt)
    try:
        cuerpo_resultado = pp.Group(pp.ZeroOrMore(sentencia)).parse_string(cuerpo_normalizado, parse_all=True)
        cuerpo = lista(cuerpo_resultado[0])
    except pp.ParseBaseException:
        cuerpo = []
    ERRORES.append((linea, "dowhile", "falta punto y coma"))
    condicion_txt = re.search(r"while\s*\((.*?)\)", bruto, re.S).group(1)
    try:
        condicion = valor(expr.parse_string(condicion_txt, parse_all=True)[0])
    except pp.ParseBaseException:
        condicion = ["error", "condicion", "expresion incompleta"]
    return Nodo(["dowhile", cuerpo, condicion])
do_malo.set_parse_action(pa_do_malo)

def pa_error(s, loc, t):
    bruto = t[0].strip(); linea = pp.lineno(loc, s)
    if bruto.rstrip(";") == "__MISSING_BLOCK__":
        ERRORES.append((linea, "bloque", "falta llave de cierre"))
        return Nodo(["error", "bloque", "falta llave de cierre"])
    if not bruto: return Nodo(["error", "sentencia", "sentencia no reconocida"])
    if bruto.startswith("System.out.print"):
        contexto, desc = "print", ("falta parentesis de cierre" if bruto.count("(") > bruto.count(")") else "falta punto y coma")
    elif bruto.startswith("break"): contexto, desc = "break", "falta punto y coma"
    elif bruto.startswith("continue"): contexto, desc = "continue", "falta punto y coma"
    elif bruto.endswith(("++", "--")): contexto, desc = "incremento", "falta punto y coma"
    elif any(bruto.startswith(x + " ") for x in TIPOS):
        if re.match(r"^(?:int|float|boolean|char|String)\s+\d", bruto): contexto, desc = "sentencia", "sentencia no reconocida"
        else: contexto, desc = "declaracion", "falta punto y coma"
    elif "=" in bruto: contexto, desc = "asignacion", "falta punto y coma"
    else: contexto, desc = "sentencia", "sentencia no reconocida"
    ERRORES.append((linea, contexto, desc))
    return Nodo(["error", contexto, desc])
error_stmt = pp.Regex(r"[^\n{};]+(?:;)?").set_parse_action(pa_error)

sentencia <<= pp.MatchFirst([declaracion, asignacion, incremento, impresion, break_stmt, continue_stmt,
                             if_stmt, while_stmt, do_malo, do_stmt, for_stmt, switch_stmt, cabecera_if_mala, cabecera_while_mala, cabecera_for_mala, bloque, error_stmt])
programa = pp.ZeroOrMore(sentencia) + pp.StringEnd()

def analizar_codigo(codigo, mostrar=True):
    global ERRORES
    ERRORES = []
    codigo = codigo.lstrip("\ufeff")
    codigo = COMENTARIOS.transform_string(codigo)
    codigo = re.sub(
        r'(?m)^(\s*case\s+)(-?\d+(?:\.\d+)?|true|false|"(?:[^"\\]|\\.)*"|\'(?:[^\'\\]|\\.)*\')(\s*)(?=\n)',
        r'\1"__CASE_COLON_MISSING__":', codigo
    )
    # Si el archivo termina con bloques abiertos, se inserta un nodo de error
    # antes de cerrar artificialmente cada bloque para conservar el AST.
    balance = 0
    for caracter in codigo:
        if caracter == "{": balance += 1
        elif caracter == "}": balance -= 1
    if balance > 0:
        codigo = codigo.rstrip() + "\n" + "\n".join("__MISSING_BLOCK__;\n}" for _ in range(balance))
    try:
        resultado = programa.parse_string(codigo, parse_all=True)
        arbol = lista(resultado)
    except pp.ParseBaseException as exc:
        ERRORES.append((exc.lineno, "sentencia", "error sintactico no recuperado"))
        arbol = []
    if mostrar:
        if ERRORES:
            print("Errores de sintaxis")
            for linea, contexto, desc in sorted(ERRORES):
                print(f"ERROR SINTACTICO (linea {linea}): {contexto} - {desc}")
        else:
            print("Código Válido")
        print("\nÁrbol de Análisis Sintáctico")
        import pprint
        pprint.pprint(arbol, width=100, sort_dicts=False)
    return arbol, sorted(ERRORES)

# Carga interactiva de un archivo .txt en Google Colab o entorno local.
def cargar_archivo():
    try:
        from google.colab import files
        subidos = files.upload()
        if not subidos:
            raise ValueError("No se selecciono ningun archivo.")
        nombre = next(iter(subidos))
        if not nombre.lower().endswith(".txt"):
            raise ValueError("El archivo debe tener extension .txt.")
        return subidos[nombre].decode("utf-8-sig")
    except ImportError:
        nombre = input("Ingrese la ruta del archivo .txt: ").strip()
        if not nombre.lower().endswith(".txt"):
            raise ValueError("El archivo debe tener extension .txt.")
        with open(nombre, "r", encoding="utf-8-sig") as archivo:
            return archivo.read()

# Ejecucion principal: al ejecutar la celda, se solicita el archivo fuente .txt.
if __name__ == "__main__":
    try:
        codigo_fuente = cargar_archivo()
        analizar_codigo(codigo_fuente)
    except Exception as error:
        print("No fue posible analizar el archivo:", error)
